In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ProjetRetardsVols") \
    .config("spark.jars.packages", "com.datastax.spark:spark-cassandra-connector_2.12:3.4.1") \
    .config("spark.cassandra.connection.host", "cassandra") \
    .config("spark.sql.extensions", "com.datastax.spark.connector.CassandraSparkExtensions") \
    .getOrCreate()

spark

In [8]:
df = spark.read.csv("/home/jovyan/data/DelayedFlights.csv", header=True, inferSchema=True)

print("Nombre de lignes :", df.count())
df.printSchema()

Nombre de lignes : 1936758
root
 |-- _c0: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DayofMonth: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- DepTime: double (nullable = true)
 |-- CRSDepTime: integer (nullable = true)
 |-- ArrTime: double (nullable = true)
 |-- CRSArrTime: integer (nullable = true)
 |-- UniqueCarrier: string (nullable = true)
 |-- FlightNum: integer (nullable = true)
 |-- TailNum: string (nullable = true)
 |-- ActualElapsedTime: double (nullable = true)
 |-- CRSElapsedTime: double (nullable = true)
 |-- AirTime: double (nullable = true)
 |-- ArrDelay: double (nullable = true)
 |-- DepDelay: double (nullable = true)
 |-- Origin: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- Distance: integer (nullable = true)
 |-- TaxiIn: double (nullable = true)
 |-- TaxiOut: double (nullable = true)
 |-- Cancelled: integer (nullable = true)
 |-- CancellationCode: string (nul

In [9]:
# On sélectionne uniquement les colonnes utiles à nos 3 indicateurs
colonnes_utiles = [
    "Year", "Month", "DayofMonth", "DayOfWeek",
    "UniqueCarrier", "Origin", "Dest",
    "ArrDelay", "DepDelay", "Cancelled"
]

df_clean = df.select(colonnes_utiles)

# On réduit à 150 000 lignes (échantillon aléatoire pour rester représentatif)
fraction = 150000 / df_clean.count()
df_sample = df_clean.sample(withReplacement=False, fraction=fraction, seed=42)

print("Nombre de lignes après échantillonnage :", df_sample.count())
df_sample.show(5)

Nombre de lignes après échantillonnage : 150005
+----+-----+----------+---------+-------------+------+----+--------+--------+---------+
|Year|Month|DayofMonth|DayOfWeek|UniqueCarrier|Origin|Dest|ArrDelay|DepDelay|Cancelled|
+----+-----+----------+---------+-------------+------+----+--------+--------+---------+
|2008|    1|         3|        4|           WN|   IND| MCO|    80.0|    94.0|        0|
|2008|    1|         3|        4|           WN|   ISP| BWI|    14.0|    25.0|        0|
|2008|    1|         3|        4|           WN|   ISP| FLL|     4.0|    29.0|        0|
|2008|    1|         3|        4|           WN|   LAS| BDL|    49.0|    59.0|        0|
|2008|    1|         3|        4|           WN|   LAS| BUR|    55.0|    51.0|        0|
+----+-----+----------+---------+-------------+------+----+--------+--------+---------+
only showing top 5 rows



### diagnostic du dataset

In [13]:
from pyspark.sql import functions as F

total = df_sample.count()
print("Total lignes :", total)

# 1. Vols annulés
nb_annules = df_sample.filter(F.col("Cancelled") == 1).count()
print(f"Vols annulés : {nb_annules} ({nb_annules/total*100:.2f}%)")

# 2. Valeurs manquantes sur ArrDelay / DepDelay
nb_null_arr = df_sample.filter(F.col("ArrDelay").isNull()).count()
nb_null_dep = df_sample.filter(F.col("DepDelay").isNull()).count()
print(f"ArrDelay manquant : {nb_null_arr} ({nb_null_arr/total*100:.2f}%)")
print(f"DepDelay manquant : {nb_null_dep} ({nb_null_dep/total*100:.2f}%)")

# 3. Doublons 
nb_doublons = total - df_sample.dropDuplicates().count()
print(f"Doublons : {nb_doublons}")

# 4. Valeurs aberrantes 
df_sample.select("ArrDelay", "DepDelay").describe().show()

Total lignes : 150005
Vols annulés : 51 (0.03%)
ArrDelay manquant : 653 (0.44%)
DepDelay manquant : 0 (0.00%)
Doublons : 9
+-------+------------------+-----------------+
|summary|          ArrDelay|         DepDelay|
+-------+------------------+-----------------+
|  count|            149352|           150005|
|   mean| 42.29690931490707|43.26663777874071|
| stddev|57.408275343887226|54.08784831533614|
|    min|             -66.0|              6.0|
|    max|            1707.0|           1710.0|
+-------+------------------+-----------------+



### nettoyage du dataset

In [14]:
df_nettoye = df_sample \
    .filter(F.col("Cancelled") == 0) \
    .filter(F.col("ArrDelay").isNotNull()) \
    .dropDuplicates()

print("Lignes avant nettoyage :", df_sample.count())
print("Lignes après nettoyage :", df_nettoye.count())

df_nettoye.show(5)

Lignes avant nettoyage : 150005
Lignes après nettoyage : 149343
+----+-----+----------+---------+-------------+------+----+--------+--------+---------+
|Year|Month|DayofMonth|DayOfWeek|UniqueCarrier|Origin|Dest|ArrDelay|DepDelay|Cancelled|
+----+-----+----------+---------+-------------+------+----+--------+--------+---------+
|2008|    1|         3|        4|           WN|   MDW| PHL|    29.0|    57.0|        0|
|2008|    1|         3|        4|           WN|   OAK| SNA|    42.0|    46.0|        0|
|2008|    1|         6|        7|           WN|   ELP| AUS|    13.0|    16.0|        0|
|2008|    1|         6|        7|           WN|   PDX| OAK|    76.0|    90.0|        0|
|2008|    1|         7|        1|           WN|   PHX| SAT|     0.0|     9.0|        0|
+----+-----+----------+---------+-------------+------+----+--------+--------+---------+
only showing top 5 rows



In [15]:
df_nettoye.write.mode("overwrite").parquet("/home/jovyan/data/flights_clean.parquet")
print("Sauvegardé avec succès.")

Sauvegardé avec succès.
